In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

from sklearn.metrics import mean_absolute_error, mean_squared_error
from src.pipeline.data_fetcher_yahoo import YahooDataFetcher
from src.pipeline.baselines import (
    persistence_forecast,
    moving_average_forecast,
    arima_forecast,
)

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("📦 Libraries imported successfully!")
print("🔗 Project modules loaded!")


In [ ]:
# Load SPY data (1 year of daily closes)
fetcher = YahooDataFetcher(max_retries=1, retry_delay=0)
df = fetcher.fetch_ohlcv("SPY", "1d", period="365d")
close = df["close"]

print(f"📈 SPY Data Loaded:")
print(f"   • Date Range: {str(close.index[0])[:10]} to {str(close.index[-1])[:10]}")
print(f"   • Total Days: {len(close)}")
print(f"   • Price Range: ${close.min():.2f} - ${close.max():.2f}")
print(f"   • Latest Price: ${close.iloc[-1]:.2f}")

# Basic statistics
returns = close.pct_change().dropna()
print(f"\n📊 Returns Statistics:")
print(f"   • Daily Return Mean: {returns.mean():.4f} ({returns.mean()*252:.2%} annualized)")
print(f"   • Daily Return Std: {returns.std():.4f} ({returns.std()*np.sqrt(252):.2%} annualized)")
print(f"   • Skewness: {returns.skew():.3f}")
print(f"   • Kurtosis: {returns.kurtosis():.3f}")

# Store key parameters for theoretical calculations
daily_std = returns.std()
price_std = close.std()
print(f"\n🔑 Key Parameters:")
print(f"   • Price Standard Deviation (σ_price): ${price_std:.4f}")
print(f"   • Returns Standard Deviation (σ_returns): {daily_std:.4f}")


In [ ]:
def evaluate_model(true: pd.Series, pred: pd.Series, name: str):
    """Evaluate forecast model and return metrics."""
    # Align indices
    true, pred = true.align(pred, join="inner")
    
    # Check for NaN values and drop them
    if pred.isna().any():
        print(f"⚠️  {name} contains {pred.isna().sum()} NaN values, dropping them")
        mask = ~(true.isna() | pred.isna())
        true, pred = true[mask], pred[mask]
    
    if len(true) == 0:
        print(f"❌ {name:20s} | No valid data points available")
        return None
    
    mae = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    mape = np.mean(np.abs((true - pred) / true)) * 100
    
    print(f"✅ {name:15s} | MAE: {mae:7.4f}  RMSE: {rmse:7.4f}  MAPE: {mape:5.2f}%")
    
    return {
        'model': name,
        'mae': mae,
        'rmse': rmse,
        'mape': mape,
        'n_points': len(true),
        'true': true,
        'pred': pred
    }

# Run baseline forecasts
print("🔄 Running Baseline Forecasts...")
print("=" * 60)

# Persistence
p_result = evaluate_model(close.iloc[1:], persistence_forecast(close), "Persistence")

# Moving Average (5-day)
ma_result = evaluate_model(close.iloc[5:], moving_average_forecast(close, window=5), "MA(5)")

# ARIMA(1,0,0)
ar_forecast = arima_forecast(close, order=(1,0,0))
ar_result = evaluate_model(close.iloc[-len(ar_forecast):], ar_forecast, "ARIMA(1,0,0)")

# Store results
empirical_results = [p_result, ma_result, ar_result]
print("=" * 60)


In [ ]:
# Calculate theoretical benchmarks for random walk
print("🎯 Theoretical Random Walk Benchmarks")
print("=" * 50)

# For random walk, the innovation variance is the daily return variance
# But we need price-level errors, so we use price differences
price_changes = close.diff().dropna()
sigma_price_change = price_changes.std()  # Standard deviation of daily price changes
mean_price = close.mean()

print(f"📊 Data-driven parameters:")
print(f"   • σ (price changes): ${sigma_price_change:.4f}")
print(f"   • Mean price level: ${mean_price:.2f}")

# Theoretical MAE for persistence forecast on random walk
# E[|εt|] = σ * sqrt(2/π) ≈ 0.798 * σ
theoretical_mae = sigma_price_change * np.sqrt(2/np.pi)

# Theoretical RMSE for persistence forecast on random walk  
# E[εt²]^0.5 = σ
theoretical_rmse = sigma_price_change

# Theoretical MAPE (approximate)
# E[|εt|] / E[P] = (σ * sqrt(2/π)) / mean_price
theoretical_mape = (theoretical_mae / mean_price) * 100

print(f"\n🧮 Theoretical Persistence Forecast (Random Walk):")
print(f"   • MAE:  {theoretical_mae:.4f}  [σ√(2/π)]")
print(f"   • RMSE: {theoretical_rmse:.4f}  [σ]")
print(f"   • MAPE: {theoretical_mape:.2f}%  [≈ MAE/E[P]]")

# Store theoretical results
theoretical_benchmark = {
    'model': 'Theory (RW)',
    'mae': theoretical_mae,
    'rmse': theoretical_rmse, 
    'mape': theoretical_mape
}

print("=" * 50)


In [ ]:
# Create comparison table
comparison_data = []

# Add empirical results
for result in empirical_results:
    if result:  # Skip None results
        comparison_data.append({
            'Model': result['model'],
            'Type': 'Empirical',
            'MAE': result['mae'],
            'RMSE': result['rmse'],
            'MAPE': result['mape']
        })

# Add theoretical benchmark
comparison_data.append({
    'Model': 'Persistence (Theory)',
    'Type': 'Theoretical',
    'MAE': theoretical_benchmark['mae'],
    'RMSE': theoretical_benchmark['rmse'],
    'MAPE': theoretical_benchmark['mape']
})

comparison_df = pd.DataFrame(comparison_data)

print("📋 Theory vs Practice Comparison Table:")
print("=" * 70)
print(f"{'Model':<20} {'Type':<12} {'MAE':<8} {'RMSE':<8} {'MAPE':<8}")
print("-" * 70)

for _, row in comparison_df.iterrows():
    print(f"{row['Model']:<20} {row['Type']:<12} {row['MAE']:<8.4f} {row['RMSE']:<8.4f} {row['MAPE']:<8.2f}%")

# Calculate ratios for persistence model
if p_result:
    mae_ratio = p_result['mae'] / theoretical_mae
    rmse_ratio = p_result['rmse'] / theoretical_rmse
    mape_ratio = p_result['mape'] / theoretical_mape
    
    print(f"\n🔍 Persistence Model: Empirical/Theoretical Ratios:")
    print(f"   • MAE Ratio:  {mae_ratio:.3f}  ({'Higher' if mae_ratio > 1 else 'Lower'} than theory)")
    print(f"   • RMSE Ratio: {rmse_ratio:.3f}  ({'Higher' if rmse_ratio > 1 else 'Lower'} than theory)")
    print(f"   • MAPE Ratio: {mape_ratio:.3f}  ({'Higher' if mape_ratio > 1 else 'Lower'} than theory)")

print("=" * 70)


In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('📊 Baseline Forecast Analysis: Theory vs Practice', fontsize=16, fontweight='bold')

# 1. MAE Comparison
ax1 = axes[0, 0]
models = [row['Model'] for row in comparison_data]
mae_values = [row['MAE'] for row in comparison_data]
colors = ['lightblue' if 'Theory' not in model else 'red' for model in models]

bars1 = ax1.bar(models, mae_values, color=colors, alpha=0.7, edgecolor='black')
ax1.set_title('Mean Absolute Error (MAE)', fontweight='bold')
ax1.set_ylabel('MAE ($)')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars1, mae_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

# 2. RMSE Comparison  
ax2 = axes[0, 1]
rmse_values = [row['RMSE'] for row in comparison_data]

bars2 = ax2.bar(models, rmse_values, color=colors, alpha=0.7, edgecolor='black')
ax2.set_title('Root Mean Square Error (RMSE)', fontweight='bold')
ax2.set_ylabel('RMSE ($)')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars2, rmse_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
             f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

# 3. MAPE Comparison
ax3 = axes[1, 0]
mape_values = [row['MAPE'] for row in comparison_data]

bars3 = ax3.bar(models, mape_values, color=colors, alpha=0.7, edgecolor='black')
ax3.set_title('Mean Absolute Percentage Error (MAPE)', fontweight='bold')
ax3.set_ylabel('MAPE (%)')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars3, mape_values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
             f'{value:.2f}%', ha='center', va='bottom', fontweight='bold')

# 4. Time Series with Forecasts (last 50 days)
ax4 = axes[1, 1]
n_show = 50

# Plot actual prices
actual_recent = close[-n_show:]
ax4.plot(actual_recent.index, actual_recent.values, 'b-', linewidth=2, label='Actual SPY', alpha=0.8)

# Plot persistence forecast
if p_result and len(p_result['pred']) >= n_show:
    pers_recent = p_result['pred'][-n_show:]
    ax4.plot(pers_recent.index, pers_recent.values, 'r--', linewidth=2, label='Persistence', alpha=0.7)

# Plot MA forecast  
if ma_result and len(ma_result['pred']) >= n_show:
    ma_recent = ma_result['pred'][-n_show:]
    ax4.plot(ma_recent.index, ma_recent.values, 'g-.', linewidth=2, label='MA(5)', alpha=0.7)

ax4.set_title('SPY Price vs Forecasts (Last 50 Days)', fontweight='bold')
ax4.set_ylabel('Price ($)')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Summary insights
print("\n💡 Key Insights:")
print("=" * 50)
print("🎯 Red bars show theoretical random walk benchmarks")
print("🔵 Blue bars show empirical model performance")
print("✅ Lower values = better forecast accuracy")


In [ ]:
# Statistical tests for random walk hypothesis
print("🔬 Statistical Tests for Random Walk Hypothesis")
print("=" * 60)

# 1. Ljung-Box Test for autocorrelation in returns
from scipy.stats import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox

returns = close.pct_change().dropna()

# Test for autocorrelation (should be NOT significant for random walk)
ljung_box = acorr_ljungbox(returns, lags=10, return_df=True)
lb_significant = (ljung_box['lb_pvalue'] < 0.05).any()

print(f"📊 Ljung-Box Test (H0: No autocorrelation):")
print(f"   • Any significant lags (p<0.05): {'Yes' if lb_significant else 'No'}")
print(f"   • Conclusion: {'Reject' if lb_significant else 'Fail to reject'} random walk hypothesis")

# 2. Variance Ratio Test (simplified)
# Compare variance of 2-day returns vs 2x variance of 1-day returns
returns_1d = returns
returns_2d = close.pct_change(2).dropna()

var_1d = returns_1d.var()
var_2d = returns_2d.var()
variance_ratio = var_2d / (2 * var_1d)

print(f"\n📈 Variance Ratio Test:")
print(f"   • 2-day variance / (2 × 1-day variance): {variance_ratio:.3f}")
print(f"   • Expected for random walk: 1.000")
print(f"   • Deviation from RW: {abs(variance_ratio - 1.0):.3f}")

# 3. Jarque-Bera test for normality
jb_stat, jb_pvalue = jarque_bera(returns)
is_normal = jb_pvalue > 0.05

print(f"\n🔔 Jarque-Bera Normality Test:")
print(f"   • Test statistic: {jb_stat:.3f}")
print(f"   • P-value: {jb_pvalue:.4f}")
print(f"   • Returns are normal: {'Yes' if is_normal else 'No'} (p>0.05)")

# Summary assessment
print(f"\n🎯 Random Walk Assessment:")
print("=" * 40)
evidence_count = 0
if not lb_significant:
    evidence_count += 1
    print("✅ No significant autocorrelation")
else:
    print("❌ Significant autocorrelation detected")
    
if abs(variance_ratio - 1.0) < 0.1:
    evidence_count += 1
    print("✅ Variance ratio close to 1.0")
else:
    print("❌ Variance ratio deviates from 1.0")

if evidence_count >= 1:
    print("\n🏆 CONCLUSION: SPY exhibits strong random walk characteristics")
    print("   → Persistence forecast is theoretically optimal")
else:
    print("\n⚠️  CONCLUSION: Some evidence against pure random walk")
    print("   → Room for more sophisticated forecasting models")

print("=" * 60)


Tasks:
	1.	Take the empirical MAE/RMSE/MAPE outputs from scripts/backtest_baselines.py.
	2.	Derive theoretical benchmarks for a naïve persistence model on a random‐walk price (e.g., expected MAE ≈σ√(2/π)).
	3.	Compare theory vs practice in a Jupyter notebook, with inline charts and Markdown commentary.
	4.	Push the notebook into notebooks/ so it’s versioned.